Load and Preprocess the Dataset

In [27]:
import pandas as pd
import numpy as np
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler

# Load dataset (adjust path if needed or use synthetic data)
try:
    df = pd.read_csv('data/expense_data_2.csv')
except FileNotFoundError:
    # Create synthetic data if file not found (example structure)
    data = {
        'Income': np.random.uniform(20000, 120000, 1000),
        'Rent': np.random.uniform(500, 5000, 1000),
        'Insurance': np.random.uniform(100, 1000, 1000),
        'Groceries': np.random.uniform(200, 2000, 1000),
        'Transport': np.random.uniform(100, 1500, 1000),
        'Eating_Out': np.random.uniform(50, 1000, 1000),
        'Entertainment': np.random.uniform(50, 1000, 1000),
        'Utilities': np.random.uniform(100, 1000, 1000),
        'Healthcare': np.random.uniform(50, 1000, 1000),
        'Miscellaneous': np.random.uniform(50, 500, 1000)
    }
    df = pd.DataFrame(data)

# Drop unnecessary targets if they exist
cols_to_drop = ["Disposable_Income", "Loan_Repayment"]
for col in cols_to_drop:
    if col in df.columns:
        df = df.drop(columns=[col])

# Fill missing numeric values
numeric_cols = df.select_dtypes(include='number').columns
for col in numeric_cols:
    df[col] = df[col].fillna(df[col].median())

# Add income transformations
if 'Income' in df.columns:
    df["Income_Log"] = np.log1p(df["Income"])
    df["Income_Squared"] = df["Income"] ** 2

# Define input and output columns
expense_columns = [
    "Rent", "Insurance", "Groceries", "Transport",
    "Eating_Out", "Entertainment", "Utilities",
    "Healthcare", "Miscellaneous"
]
input_columns = ["Income", "Income_Log", "Income_Squared"] + expense_columns
X = df[input_columns]
y = df[expense_columns]

# Train-test split
X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=42)

# Scale data for LSTM, ANN, and SVR
scaler = StandardScaler()
X_train_scaled = scaler.fit_transform(X_train)
X_test_scaled = scaler.transform(X_test)


Train and Evaluate SpendSense’s XGBoost Model

In [28]:
from xgboost import XGBRegressor
from sklearn.multioutput import MultiOutputRegressor
from sklearn.metrics import mean_absolute_error, r2_score
import joblib

# Train SpendSense's XGBoost model
xgb = XGBRegressor(
    n_estimators=300,
    learning_rate=0.03,
    max_depth=4,
    subsample=0.8,
    colsample_bytree=0.8,
    reg_alpha=0.4,
    reg_lambda=0.4,
    random_state=42
)
model_xgb = MultiOutputRegressor(xgb)
model_xgb.fit(X_train, y_train)  # No scaling needed for XGBoost

# Predict and evaluate
y_pred_xgb = model_xgb.predict(X_test)
r2_xgb = r2_score(y_test, y_pred_xgb, multioutput='raw_values')
mae_xgb = mean_absolute_error(y_test, y_pred_xgb, multioutput='raw_values')

# Save model if needed
joblib.dump(model_xgb, "unified_expense_predictor.pkl")

# Print results for reference
print("SpendSense XGBoost Results:")
for i, col in enumerate(expense_columns):
    print(f"{col}: R² = {r2_xgb[i]:.4f}, MAE = {mae_xgb[i]:.2f}")


SpendSense XGBoost Results:
Rent: R² = 0.9427, MAE = 229.95
Insurance: R² = 0.9372, MAE = 42.72
Groceries: R² = 0.9496, MAE = 133.06
Transport: R² = 0.9073, MAE = 79.03
Eating_Out: R² = 0.9812, MAE = 37.64
Entertainment: R² = 0.9382, MAE = 38.62
Utilities: R² = 0.9701, MAE = 58.02
Healthcare: R² = 0.9621, MAE = 44.35
Miscellaneous: R² = 0.9714, MAE = 21.20


Train and Evaluate an MLP Model

In [29]:
from tensorflow.keras.models import Sequential
from tensorflow.keras.layers import Dense, Input

# MLP model
model_mlp = Sequential()
model_mlp.add(Input(shape=(X_train_scaled.shape[1],)))
model_mlp.add(Dense(128, activation='relu'))
model_mlp.add(Dense(64, activation='relu'))
model_mlp.add(Dense(len(expense_columns)))  # Output layer

model_mlp.compile(optimizer='adam', loss='mse')

# Train model
model_mlp.fit(X_train_scaled, y_train, epochs=50, batch_size=32, verbose=0, validation_split=0.2)

# Predict and evaluate
y_pred_mlp = model_mlp.predict(X_test_scaled)
r2_mlp = r2_score(y_test, y_pred_mlp, multioutput='raw_values')
mae_mlp = mean_absolute_error(y_test, y_pred_mlp, multioutput='raw_values')

print("MLP Results:")
for i, col in enumerate(expense_columns):
    print(f"{col}: R² = {r2_mlp[i]:.4f}, MAE = {mae_mlp[i]:.2f}")


125/125 ━━━━━━━━━━━━━━━━━━━━ 0s 820us/step
MLP Results:
Rent: R² = 0.9998, MAE = 37.87
Insurance: R² = 0.9191, MAE = 273.97
Groceries: R² = 0.9848, MAE = 190.40
Transport: R² = 0.9708, MAE = 301.37
Eating_Out: R² = 0.8866, MAE = 224.65
Entertainment: R² = 0.9083, MAE = 277.41
Utilities: R² = 0.8989, MAE = 384.50
Healthcare: R² = 0.9534, MAE = 204.75
Miscellaneous: R² = 0.9067, MAE = 166.66


 Train and Evaluate ANN Model

In [30]:
from tensorflow.keras.models import Sequential
from tensorflow.keras.layers import Dense
from sklearn.metrics import mean_absolute_error, r2_score

# Build ANN model for multi-output regression
model_ann = Sequential()
model_ann.add(Dense(64, activation='relu', input_dim=X_train_scaled.shape[1]))
model_ann.add(Dense(32, activation='relu'))
model_ann.add(Dense(len(expense_columns)))  # Output layer for multiple expense categories
model_ann.compile(optimizer='adam', loss='mse')

# Train model
model_ann.fit(X_train_scaled, y_train, epochs=50, batch_size=32, verbose=0, validation_split=0.2)

# Predict and evaluate
y_pred_ann = model_ann.predict(X_test_scaled)
r2_ann = r2_score(y_test, y_pred_ann, multioutput='raw_values')
mae_ann = mean_absolute_error(y_test, y_pred_ann, multioutput='raw_values')

# Print results
print("ANN Results:")
for i, col in enumerate(expense_columns):
    print(f"{col}: R² = {r2_ann[i]:.4f}, MAE = {mae_ann[i]:.2f}")


c:\Users\Ravindu\AppData\Local\Programs\Python\Python310\lib\site-packages\keras\src\layers\core\dense.py:87: UserWarning: Do not pass an `input_shape`/`input_dim` argument to a layer. When using Sequential models, prefer using an `Input(shape)` object as the first layer in the model instead.
  super().__init__(activity_regularizer=activity_regularizer, **kwargs)


125/125 ━━━━━━━━━━━━━━━━━━━━ 0s 887us/step
ANN Results:
Rent: R² = 0.9982, MAE = 67.60
Insurance: R² = 0.8983, MAE = 310.39
Groceries: R² = 0.9768, MAE = 314.38
Transport: R² = 0.9723, MAE = 297.88
Eating_Out: R² = 0.8347, MAE = 299.26
Entertainment: R² = 0.8946, MAE = 307.04
Utilities: R² = 0.9103, MAE = 389.35
Healthcare: R² = 0.9501, MAE = 209.45
Miscellaneous: R² = 0.8537, MAE = 202.31


Train and Evaluate SVR Model

In [31]:
from sklearn.svm import SVR
from sklearn.multioutput import MultiOutputRegressor
from sklearn.metrics import mean_absolute_error, r2_score

# Train SVR model (with scaling)
model_svr = MultiOutputRegressor(SVR(kernel='rbf', C=1.0, epsilon=0.1))
model_svr.fit(X_train_scaled, y_train)

# Predict and evaluate
y_pred_svr = model_svr.predict(X_test_scaled)
r2_svr = r2_score(y_test, y_pred_svr, multioutput='raw_values')
mae_svr = mean_absolute_error(y_test, y_pred_svr, multioutput='raw_values')

# Print results
print("SVR Results:")
for i, col in enumerate(expense_columns):
    print(f"{col}: R² = {r2_svr[i]:.4f}, MAE = {mae_svr[i]:.2f}")


SVR Results:
Rent: R² = 0.1441, MAE = 3862.96
Insurance: R² = 0.3823, MAE = 407.88
Groceries: R² = 0.2727, MAE = 1592.36
Transport: R² = 0.3213, MAE = 722.73
Eating_Out: R² = 0.4369, MAE = 405.53
Entertainment: R² = 0.3843, MAE = 401.12
Utilities: R² = 0.3588, MAE = 721.14
Healthcare: R² = 0.4227, MAE = 398.57
Miscellaneous: R² = 0.4737, MAE = 208.08


Results Comparison

In [32]:
import pandas as pd
import numpy as np

# Average metrics across categories
results_summary = {
    'Model': ['XGBoost (SpendSense)', 'MLP', 'ANN', 'SVR'],
    'Avg_R2': [np.mean(r2_xgb), np.mean(r2_mlp), np.mean(r2_ann), np.mean(r2_svr)],
    'Avg_MAE': [np.mean(mae_xgb), np.mean(mae_mlp), np.mean(mae_ann), np.mean(mae_svr)]
}
results_df = pd.DataFrame(results_summary)
print("\nSummary of Average Performance Across All Categories:")
print(results_df.to_string(index=False))



Summary of Average Performance Across All Categories:
               Model   Avg_R2    Avg_MAE
XGBoost (SpendSense) 0.951101  76.065773
                 MLP 0.936482 229.064178
                 ANN 0.920991 266.404999
                 SVR 0.355204 968.931019


For the Stock Forecasting Module

Load and Preprocess the Data

In [44]:
import pandas as pd
import numpy as np
from sklearn.preprocessing import MinMaxScaler

# Load data (example for AAPL)
symbol = "AAPL"
df = pd.read_csv(f"data/stock/prophet/{symbol}_stock.csv", skiprows=3)
df.columns = ["Date", "Close", "High", "Low", "Open", "Volume"]
df["Date"] = pd.to_datetime(df["Date"])
df = df.sort_values("Date").reset_index(drop=True)

# Prophet data
prophet_df = df[["Date", "Close"]].rename(columns={"Date": "ds", "Close": "y"})

# LSTM data: Scale and create sequences
scaler = MinMaxScaler()
scaled_data = scaler.fit_transform(df[["Close"]])
def create_sequences(data, seq_length):
    X, y = [], []
    for i in range(len(data) - seq_length):
        X.append(data[i:(i + seq_length)])
        y.append(data[i + seq_length])
    return np.array(X), np.array(y)
seq_length = 60
X_lstm, y_lstm = create_sequences(scaled_data, seq_length)

# ARIMA data: Use raw Close prices, check for stationarity if needed
arima_data = df["Close"]


 Train and Predict with Each Model

Prophet

In [45]:
# Prophet forecast (from your notebook)
prophet_model = Prophet(daily_seasonality=True)
prophet_model.fit(prophet_df)
future_prophet = prophet_model.make_future_dataframe(periods=180)
forecast_prophet = prophet_model.predict(future_prophet)

# Split into training and test for evaluation (last 180 days of historical data as test)
train_size = len(df) - 180
train_prophet = df[:train_size]
test_prophet = df[train_size:]
prophet_pred = forecast_prophet[forecast_prophet['ds'].isin(test_prophet['Date'])]['yhat'].values
prophet_actual = test_prophet['Close'].values
prophet_mae = mean_absolute_error(prophet_actual, prophet_pred)
prophet_rmse = np.sqrt(mean_squared_error(prophet_actual, prophet_pred))
print(f"Prophet MAE: {prophet_mae:.2f}, RMSE: {prophet_rmse:.2f}")


03:11:04 - cmdstanpy - INFO - Chain [1] start processing
03:11:04 - cmdstanpy - INFO - Chain [1] done processing


Prophet MAE: 6.42, RMSE: 8.41


LSTM

In [46]:
# Prepare data for LSTM
scaler = MinMaxScaler()
scaled_data = scaler.fit_transform(df[['Close']])

def create_sequences(data, seq_length):
    X, y = [], []
    for i in range(len(data) - seq_length):
        X.append(data[i:(i + seq_length)])
        y.append(data[i + seq_length])
    return np.array(X), np.array(y)

seq_length = 60
X_lstm, y_lstm = create_sequences(scaled_data, seq_length)

# Split into train and test
train_size_lstm = len(X_lstm) - 180
X_train_lstm, X_test_lstm = X_lstm[:train_size_lstm], X_lstm[train_size_lstm:]
y_train_lstm, y_test_lstm = y_lstm[:train_size_lstm], y_lstm[train_size_lstm:]

# Build LSTM model
lstm_model = Sequential()
lstm_model.add(LSTM(units=50, return_sequences=True, input_shape=(seq_length, 1)))
lstm_model.add(LSTM(units=50))
lstm_model.add(Dense(1))
lstm_model.compile(optimizer='adam', loss='mse')

# Train model
lstm_model.fit(X_train_lstm, y_train_lstm, epochs=50, batch_size=32, verbose=0)

# Predict on test set
lstm_pred_scaled = lstm_model.predict(X_test_lstm, verbose=0)
lstm_pred = scaler.inverse_transform(lstm_pred_scaled)
lstm_actual = scaler.inverse_transform(y_test_lstm)

# Calculate metrics
lstm_mae = mean_absolute_error(lstm_actual, lstm_pred)
lstm_rmse = np.sqrt(mean_squared_error(lstm_actual, lstm_pred))
print(f"LSTM MAE: {lstm_mae:.2f}, RMSE: {lstm_rmse:.2f}")


LSTM MAE: 4.17, RMSE: 5.15


ARIMA

In [47]:
# Prepare data for ARIMA (check stationarity and difference if needed)
from statsmodels.tsa.stattools import adfuller

result = adfuller(df['Close'])
if result[1] > 0.05:  # Non-stationary if p-value > 0.05
    arima_data = df['Close'].diff().dropna()
else:
    arima_data = df['Close']

# Split into train and test
train_arima = arima_data[:train_size]
test_arima = df['Close'][train_size:]

# Fit ARIMA model (order can be tuned for better results)
arima_model = ARIMA(train_arima, order=(1,1,1))
arima_fit = arima_model.fit()

# Forecast
arima_pred = arima_fit.forecast(steps=len(test_arima))
if result[1] > 0.05:
    # Reverse differencing if data was differenced
    arima_pred = df['Close'][train_size-1:train_size].values[0] + arima_pred.cumsum()

# Calculate metrics
arima_mae = mean_absolute_error(test_arima, arima_pred)
arima_rmse = np.sqrt(mean_squared_error(test_arima, arima_pred))
print(f"ARIMA MAE: {arima_mae:.2f}, RMSE: {arima_rmse:.2f}")


ARIMA MAE: 12.26, RMSE: 15.65


In [48]:
import plotly.graph_objects as go

fig = go.Figure()
# Actual data
fig.add_trace(go.Scatter(x=test_prophet['Date'], y=test_prophet['Close'], mode='lines', name='Actual', line=dict(color='black')))
# Prophet predictions
fig.add_trace(go.Scatter(x=test_prophet['Date'], y=prophet_pred, mode='lines', name='Prophet Predicted', line=dict(color='blue')))
# LSTM predictions
fig.add_trace(go.Scatter(x=test_prophet['Date'][-len(lstm_pred):], y=lstm_pred.flatten(), mode='lines', name='LSTM Predicted', line=dict(color='green')))
# ARIMA predictions
fig.add_trace(go.Scatter(x=test_prophet['Date'], y=arima_pred, mode='lines', name='ARIMA Predicted', line=dict(color='red')))
fig.update_layout(title='Stock Price Prediction Comparison for AAPL', xaxis_title='Date', yaxis_title='Price')
fig.show()
